# Gap 3 — Evidence normalization (Ollama × 3 model, chạy trên Kaggle)

Chỉ phần **Ollama** chạy ở đây (cần GPU). GPT-3.5 và toàn bộ phần evaluation chạy local — xem `docs/gap3_runbook.md`.

**Chuẩn bị (1 lần):**
1. Upload `bird-gap3-src.zip` (thư mục gốc repo) thành Kaggle dataset (vd. `bird-gap3`). Kaggle tự giải nén ra 4 file: `ollama_request.py`, `ollama_request2.py`, `evidence_norm.py`, `changed_ids.json`.
2. *Add Input*: dataset gap3 ở trên **và** dataset BIRD (`bird-sql`).
3. *Settings*: Accelerator = **GPU T4 x2**, **Internet = On**.

**Kế hoạch chạy** (mỗi model: pull → đối chứng 182 câu → C 182 câu → xoá model để giải phóng ổ):

| Cell | Việc | Bắt buộc |
|---|---|---|
| 1–2 | Setup + cài Ollama | ✅ |
| 3 | Smoke test 5 câu (1 model) | ✅ |
| 4 | **Gap 3**: ctrl182 + norm182 cho 3 model | ✅ |
| 5 | Đóng gói `gap3_results.zip` | ✅ |
| 6 | Baseline full 1534 câu × 3 model (chạy lại vì bug prompt) | tuỳ quota GPU |

Mọi lượt chạy đều **bỏ qua nếu đã xong** (file output đủ số câu), nên nếu session bị ngắt chỉ cần chạy lại cell — không mất phần đã làm (miễn là chưa tắt session; nên tải zip về sau cell 4).

In [ ]:
# 1. Tìm dữ liệu BIRD + file của bird-gap3-src, copy vào /kaggle/working
import os, shutil, glob

BIRD_ROOT = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'dev.json' in files and os.path.basename(root) == 'data':
        BIRD_ROOT = os.path.dirname(root)  # thư mục cha của "data/"
        break
assert BIRD_ROOT, ('Không tìm thấy bird/data/dev.json — chưa attach dataset BIRD (bird-sql). '
                   f'Đang attach: {os.listdir("/kaggle/input")}')

LOCAL_DB_ROOT = '/kaggle/working/dev_databases'
if not os.path.exists(LOCAL_DB_ROOT):
    shutil.copytree(f'{BIRD_ROOT}/data/dev_databases', LOCAL_DB_ROOT)

# lấy cả 4 file từ CÙNG thư mục chứa evidence_norm.py (tránh nhặt nhầm ollama_request.py cũ trong dataset BIRD)
hits = glob.glob('/kaggle/input/**/evidence_norm.py', recursive=True)
assert hits, 'Không thấy evidence_norm.py — attach dataset bird-gap3-src chưa?'
SRC_DIR = os.path.dirname(hits[0])

WORK = '/kaggle/working/gap3'
os.makedirs(WORK, exist_ok=True)
for name in ['ollama_request.py', 'ollama_request2.py', 'evidence_norm.py', 'changed_ids.json']:
    assert os.path.exists(f'{SRC_DIR}/{name}'), f'Thiếu {name} trong {SRC_DIR}'
    shutil.copy(f'{SRC_DIR}/{name}', WORK)

import json
DEV = f'{BIRD_ROOT}/data/dev.json'
OUT = '/kaggle/working/exp_result/gap3'
print('BIRD_ROOT =', BIRD_ROOT)
print('SRC_DIR   =', SRC_DIR)
print('DBs:', len(os.listdir(LOCAL_DB_ROOT)), '| changed_ids:', len(json.load(open(f'{WORK}/changed_ids.json'))))

In [ ]:
# 2. Cài + khởi động Ollama
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
ollama_proc = subprocess.Popen(['ollama', 'serve'])
time.sleep(5)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Danh sách model (tag Ollama, tên viết tắt dùng trong tên thư mục output)
MODELS = [
    ('deepseek-coder-v2:16b',     'deepseek'),
    ('gemma3:12b',                'gemma3'),
    ('qwen2.5-coder:7b-instruct', 'qwen25'),
]
N_CHANGED = len(json.load(open(f'{WORK}/changed_ids.json')))
N_ALL = len(json.load(open(DEV)))
sh = get_ipython().system  # chạy lệnh shell, output hiện ngay trong cell

def done(path, n):
    """True nếu file output đã có đủ n câu (để chạy lại cell không làm lại từ đầu)."""
    try:
        return len(json.load(open(path))) >= n
    except Exception:
        return False

def run(model, tag, name, extra, n):
    out = f'{OUT}/{tag}_{name}/predict_dev.json'
    if done(out, n):
        print(f'[skip] {out} đã đủ {n} câu')
        return
    print(f'########## {model} -> {tag}_{name} ##########')
    sh(f'cd {WORK} && python ollama_request2.py --model {model} --eval_path {DEV} '
       f'--db_root_path {LOCAL_DB_ROOT}/ {extra} --data_output_path {out}')
    print('->', out, 'OK' if done(out, n) else 'CHƯA ĐỦ (xem log bên trên)')

print('N_CHANGED =', N_CHANGED, '| N_ALL =', N_ALL)


In [ ]:
# 3. Smoke test — 5 câu với model đầu tiên. Dòng `-- External Knowledge` phải có CAST(... AS REAL) / ..., không còn DIVIDE(
model, tag = MODELS[0]
!ollama pull {model}
!cd {WORK} && python ollama_request2.py --model {model} --eval_path {DEV} --db_root_path {LOCAL_DB_ROOT}/     --ids_file changed_ids.json --limit 5 --normalize_evidence --print_prompts 5     --data_output_path {OUT}/smoke/predict_dev.json

In [ ]:
# 4. Gap 3 — với mỗi model: đối chứng (prompt gốc) + C (evidence chuẩn hoá), cùng 182 câu
import time
for model, tag in MODELS:
    t0 = time.time()
    sh(f'ollama pull {model}')
    run(model, tag, 'kg_ctrl182', '--ids_file changed_ids.json', N_CHANGED)
    run(model, tag, 'kg_norm182', '--ids_file changed_ids.json --normalize_evidence', N_CHANGED)
    sh(f'ollama rm {model}')
    print(f'=== {tag}: {(time.time() - t0) / 60:.1f} phút ===')


In [ ]:
# 5. Đóng gói kết quả — tải gap3_results.zip ở panel Output, giải nén vào src/DAMO-ConvAI/bird/llm/ ở local
!cd /kaggle/working && rm -rf exp_result/gap3/smoke && rm -f gap3_results.zip && zip -qr gap3_results.zip exp_result/gap3 && unzip -l gap3_results.zip

## 6. (Tuỳ chọn) Baseline full 1534 câu × 3 model
Chỉ cần nếu muốn có EX toàn tập cho Ollama (đồng thời là lượt chạy lại vì bug prompt trong PROJECT_LOG mục 9). Mỗi model mất vài giờ — kiểm tra quota GPU còn lại trước. Có thể chạy từng model một bằng cách sửa `MODELS[...]`. Chạy xong nhớ chạy lại cell 5 để đóng gói.

In [ ]:
for model, tag in MODELS:
    sh(f'ollama pull {model}')
    run(model, tag, 'kg_base', '', N_ALL)
    sh(f'ollama rm {model}')
